In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.3 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [3]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024 ** 3)

print(f"CPU RAM: {ram_gb:.1f} GB")

CPU RAM: 12.7 GB


In [4]:
import os
import torch
import pandas as pd
import numpy as np

from google.colab import drive

drive.mount("/content/drive")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

Mounted at /content/drive
GPU: Tesla T4
CUDA: True


In [5]:
train_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
test_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

TARGET_INTENTS = [
    "card_arrival",
    "card_not_working",
    "declined_card_payment",
    "lost_or_stolen_card",
    "change_pin",
    "activate_my_card",
    "pending_transfer",
    "failed_transfer",
    "declined_transfer",
    "Refund_not_showing_up",
    "cash_withdrawal_not_recognised",
    "exchange_rate",
    "top_up_by_card_charge",
    "wrong_amount_of_cash_received",
    "verify_my_identity",
]

generation_df = train_df[
    train_df["category"].isin(TARGET_INTENTS)
].copy()

print("Full training data:", train_df.shape)
print("Generation data:", generation_df.shape)
print("Number of target intents:", generation_df["category"].nunique())

Full training data: (10003, 2)
Generation data: (2031, 2)
Number of target intents: 15


In [6]:
INTENT_RESPONSES = {
    "card_arrival": "I'm sorry your card has not arrived yet. Please check your card delivery or tracking information. If the expected delivery period has passed, please contact customer support for further assistance.",

    "card_not_working": "I'm sorry your card is not working. Please check whether the card is activated and whether there are any restrictions on it. If the problem continues, please contact customer support.",

    "declined_card_payment": "I'm sorry your card payment was declined. Please check that your card is active and that you have sufficient available funds. If the issue continues, please contact customer support.",

    "lost_or_stolen_card": "If your card has been lost or stolen, please secure the card as soon as possible using your banking service and contact customer support for further assistance.",

    "change_pin": "You can change your card PIN through the available PIN-management option in your banking service. Follow the instructions provided and contact customer support if you need further assistance.",

    "activate_my_card": "To activate your card, use the card activation option provided by your banking service and follow the instructions. If activation does not work, please contact customer support.",

    "pending_transfer": "Your transfer may still be processing. Please check the transfer status in your banking service and allow the transaction time to complete. If it remains pending, contact customer support.",

    "failed_transfer": "I'm sorry your transfer failed. Please check the transfer details and try again if appropriate. If the transfer continues to fail, please contact customer support.",

    "declined_transfer": "I'm sorry your transfer was declined. Please check the transfer details and available funds. If the problem continues, please contact customer support for assistance.",

    "Refund_not_showing_up": "I'm sorry your refund is not showing up yet. Please check the original transaction and refund status. If the expected refund has not appeared, please contact customer support.",

    "cash_withdrawal_not_recognised": "If you do not recognise a cash withdrawal, please review the transaction details and contact your bank's support team as soon as possible to report the transaction.",

    "exchange_rate": "Exchange rates can vary depending on the transaction and currency pair. Please check the current exchange rate shown by your banking service before completing the transaction.",

    "top_up_by_card_charge": "If you were charged unexpectedly when topping up by card, please review the top-up transaction details. If the charge is still unclear, contact customer support for assistance.",

    "wrong_amount_of_cash_received": "I'm sorry the cash amount you received was incorrect. Please keep the transaction details and contact customer support so the withdrawal can be reviewed.",

    "verify_my_identity": "To verify your identity, follow the identity-verification steps provided by your banking service. Make sure the requested information or documents are clear and valid. Contact customer support if verification does not work."
}

generation_df["response"] = generation_df["category"].map(INTENT_RESPONSES)

generation_df = generation_df.rename(
    columns={"text": "query"}
)

generation_df = generation_df[
    ["query", "category", "response"]
].reset_index(drop=True)

print("Examples:", len(generation_df))
print("Missing responses:", generation_df["response"].isna().sum())

Examples: 2031
Missing responses: 0


In [7]:
from sklearn.model_selection import train_test_split

train_gen, temp_gen = train_test_split(
    generation_df,
    test_size=0.30,
    random_state=42,
    stratify=generation_df["category"]
)

val_gen, test_gen = train_test_split(
    temp_gen,
    test_size=0.50,
    random_state=42,
    stratify=temp_gen["category"]
)

print(f"Train: {len(train_gen)}")
print(f"Validation: {len(val_gen)}")
print(f"Test: {len(test_gen)}")

Train: 1421
Validation: 305
Test: 305


In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [9]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a helpful banking support assistant. "
    "Give concise, clear, and relevant support responses. "
    "Do not invent policies, fees, deadlines, or requirements."
)

def make_dataset(df):
    conversations = []

    for _, row in df.iterrows():
        conversations.append({
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": (
                        f"Customer intent: {row['category']}\n"
                        f"Customer query: {row['query']}"
                    )
                },
                {
                    "role": "assistant",
                    "content": row["response"]
                }
            ]
        })

    return Dataset.from_list(conversations)


train_sft = make_dataset(train_gen)
val_sft = make_dataset(val_gen)

MAX_LENGTH = 256

def tokenize_response_only(example):

    messages = example["messages"]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    labels = full["input_ids"].copy()

    prompt_length = min(
        len(prompt["input_ids"]),
        len(labels)
    )

    labels[:prompt_length] = [-100] * prompt_length

    full["labels"] = labels

    return full


train_final = train_sft.map(
    tokenize_response_only,
    remove_columns=train_sft.column_names
)

val_final = val_sft.map(
    tokenize_response_only,
    remove_columns=val_sft.column_names
)

print("Training examples:", len(train_final))
print("Validation examples:", len(val_final))

Map:   0%|          | 0/1421 [00:00<?, ? examples/s]

Map:   0%|          | 0/305 [00:00<?, ? examples/s]

Training examples: 1421
Validation examples: 305


In [10]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir="./banking-qlora-final",

    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Avoid the T4 bfloat16/fp16 conflict we hit earlier
    fp16=False,
    bf16=False,

    logging_steps=25,
    report_to="none",
    seed=42,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_final,
    eval_dataset=val_final,
    data_collator=data_collator
)

print("Trainer ready.")

Trainer ready.


In [11]:
train_result = trainer.train()

print("Training complete!")
print(f"Training loss: {train_result.training_loss:.4f}")

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.349483,0.174205


Training complete!
Training loss: 0.8232


In [12]:
from google.colab import drive

drive.mount("/content/drive")

ADAPTER_PATH = "/content/drive/MyDrive/banking-support-qlora-final"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("QLoRA adapter saved to:")
print(ADAPTER_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QLoRA adapter saved to:
/content/drive/MyDrive/banking-support-qlora-final


In [13]:
BERT_PATH = "/content/drive/MyDrive/banking77-bert-final"

from transformers import AutoTokenizer, AutoModelForSequenceClassification

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_PATH
)

bert_model.to("cuda")
bert_model.eval()

print("BERT model loaded.")
print("Number of intents:", bert_model.config.num_labels)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERT model loaded.
Number of intents: 77


In [14]:
def predict_intent(query):

    inputs = bert_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    inputs = {
        k: v.to("cuda")
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = bert_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0, predicted_id].item()

    intent = bert_model.config.id2label[predicted_id]

    return intent, confidence


print(predict_intent("My card hasn't arrived yet"))

('card_arrival', 0.825436532497406)


In [15]:
SUPPORTED_INTENTS = set(TARGET_INTENTS)

def generate_response(query, intent, max_new_tokens=80):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful banking support assistant. "
                "Give concise, clear, and relevant support responses. "
                "Do not invent policies, fees, deadlines, or requirements."
            )
        },
        {
            "role": "user",
            "content": (
                f"Customer intent: {intent}\n"
                f"Customer query: {query}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


def banking_support(query):

    intent, confidence = predict_intent(query)

    if intent not in SUPPORTED_INTENTS:
        return {
            "intent": intent,
            "confidence": confidence,
            "response": (
                "I can identify your banking query, but this prototype "
                "does not currently generate responses for this intent."
            )
        }

    response = generate_response(query, intent)

    return {
        "intent": intent,
        "confidence": confidence,
        "response": response
    }

In [16]:
test_queries = [
    "My card hasn't arrived yet",
    "My transfer is still pending",
    "Someone withdrew cash from my account and I don't recognize it",
    "I want to change my PIN",
    "My card payment was declined"
]

for query in test_queries:
    result = banking_support(query)

    print("=" * 70)
    print("Customer:", query)
    print("Intent:", result["intent"])
    print("Confidence:", round(result["confidence"], 3))
    print("Response:", result["response"])

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Customer: My card hasn't arrived yet
Intent: card_arrival
Confidence: 0.825
Response: I'm sorry your card has not arrived yet. Please check your card delivery or tracking information. If the expected delivery period has passed, please contact customer support for further assistance.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Customer: My transfer is still pending
Intent: pending_transfer
Confidence: 0.802
Response: Your transfer may be still processing. Please check the transfer status in your banking service and allow the transaction time to complete. If it remains pending, contact customer support.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Customer: Someone withdrew cash from my account and I don't recognize it
Intent: cash_withdrawal_not_recognised
Confidence: 0.867
Response: If you do not recognise a cash withdrawal, please review the transaction details and contact your bank's support team as soon as possible to report the transaction.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Customer: I want to change my PIN
Intent: change_pin
Confidence: 0.83
Response: To change your PIN, follow the instructions in your banking service. If the change does not work, please contact customer support for further assistance.
Customer: My card payment was declined
Intent: declined_card_payment
Confidence: 0.803
Response: I'm sorry your card payment was declined. Please check that your card is active and that you have sufficient available funds. If the issue continues, please contact customer support.
